# Week 8 Day 1 - Gasoline Domain Setup

Original structural reference: `notebooks_original/week8/day1.ipynb`

Week 6-7 stay on local M4 hourly data to keep forecasting discipline and benchmark comparability.
Week 8 pivots to weekly retail gasoline prices so the app/agent layer maps directly to Donner's pricing workflow.

Donner -> ForecastLLM mapping:
- product price -> gasoline price
- deal/opportunity -> forecast alert or price-change opportunity
- pricer -> forecaster
- pricing error -> forecast error
- messaging/planning agent -> forecast report/action recommendation


## Day 1 Scope

1. Load weekly gasoline price data from local CSV (`FRED/EIA` style).
2. Normalize to `timestamp` and `value` plus source metadata.
3. Build weekly lag features (`lag_1`, `lag_2`, `lag_4`, optional `lag_52`).
4. Create `ForecastCase` examples with gasoline context.
5. Show placeholder results only (no full agents yet).

Agentic workflow shape for Week 8:
- scanner -> forecaster -> evaluator -> reporter/planner


In [1]:
import os

import pandas as pd
from dotenv import load_dotenv

from week8.forecast_case import case_from_row, result_from_forecast
from week8.gasoline_loader import load_gasoline_series


In [2]:
load_dotenv()
print(f"GASOLINE_DATA_PATH={os.getenv('GASOLINE_DATA_PATH') or 'not_set'}")


GASOLINE_DATA_PATH=/home/geo/Projects/Python/forecastllm/data/gasoline/GASREGW.csv


In [3]:
gas_df = load_gasoline_series(series_id='GASREGW')

required_cols = {'timestamp', 'value', 'series_id', 'source', 'description', 'unit'}
missing = required_cols - set(gas_df.columns)
if missing:
    raise ValueError(f"Gasoline dataframe missing required columns: {sorted(missing)}")

gas_df = gas_df.copy()
gas_df['timestamp'] = pd.to_datetime(gas_df['timestamp'], errors='coerce')
gas_df['value'] = pd.to_numeric(gas_df['value'], errors='coerce')
gas_df = gas_df.dropna(subset=['timestamp', 'value']).sort_values('timestamp').reset_index(drop=True)

if len(gas_df) < 60:
    raise RuntimeError(f"Need at least 60 weekly rows for Day 1 scaffolding, got {len(gas_df)}")

print(f"Loaded {len(gas_df):,} weekly rows from {gas_df['source'].iloc[0]} ({gas_df['series_id'].iloc[0]})")
gas_df.head()


Loaded 1,857 weekly rows from FRED/EIA (GASREGW)


,timestamp,value,series_id,source,description,unit
0,1990-08-20,1.191,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon
1,1990-08-27,1.245,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon
2,1990-09-03,1.242,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon
3,1990-09-10,1.252,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon
4,1990-09-17,1.266,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon


In [4]:
supervised_df = gas_df[['timestamp', 'value', 'series_id', 'source', 'description', 'unit']].copy()
supervised_df['lag_1'] = supervised_df['value'].shift(1)
supervised_df['lag_2'] = supervised_df['value'].shift(2)
supervised_df['lag_4'] = supervised_df['value'].shift(4)
if len(supervised_df) > 52:
    supervised_df['lag_52'] = supervised_df['value'].shift(52)

supervised_df = supervised_df.dropna().reset_index(drop=True)
if supervised_df.empty:
    raise RuntimeError('No supervised rows available after lag construction.')

n = len(supervised_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
train_df = supervised_df.iloc[:train_end].copy()
val_df = supervised_df.iloc[train_end:val_end].copy()
test_df = supervised_df.iloc[val_end:].copy()

print(f"Supervised rows={n:,} | train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}")
supervised_df.head()


Supervised rows=1,805 | train=1,263, val=271, test=271


,timestamp,value,series_id,source,description,unit,lag_1,lag_2,lag_4,lag_52
0,1991-09-30,1.092,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon,1.097,1.110,1.127,1.191
1,1991-10-07,1.089,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon,1.092,1.097,1.120,1.245
2,1991-10-14,1.084,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon,1.089,1.092,1.110,1.242
3,1991-10-21,1.088,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon,1.084,1.089,1.097,1.252
4,1991-10-28,1.091,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon,1.088,1.084,1.092,1.266


## ForecastCase Setup

`ForecastCase` remains the object contract for Week 8 orchestration.

For gasoline, each case carries:
- series identity (`GASREGW`)
- weekly lag features
- observed value for evaluation
- metadata for downstream reporting/planning


In [5]:
feature_keys = ['lag_1', 'lag_2', 'lag_4']
if 'lag_52' in supervised_df.columns:
    feature_keys.append('lag_52')

sample_rows = test_df.head(5)
cases = [
    case_from_row(
        row,
        series_id=str(row.series_id),
        horizon=1,
        feature_keys=feature_keys,
        metadata={
            'split': 'test',
            'source': str(row.source),
            'commodity': 'retail gasoline',
            'fuel_type': 'regular all formulations',
            'geography': 'United States',
            'unit': str(row.unit),
            'description': str(row.description),
        },
    )
    for row in sample_rows.itertuples(index=False)
]

cases[:2]


[ForecastCase(series_id='GASREGW', horizon=1, cutoff_timestamp='2021-02-22 00:00:00', context={'day_of_week': None, 'month': None}, features={'lag_1': 2.501, 'lag_2': 2.461, 'lag_4': 2.392, 'lag_52': 2.466}, actual=2.633, naive_forecast=2.501, seasonal_naive_forecast=None, model_forecast=None, metadata={'protocol': 'one_step_ahead_chronological_no_shuffle', 'metrics': ['mae', 'smape'], 'split': 'test', 'source': 'FRED/EIA', 'commodity': 'retail gasoline', 'fuel_type': 'regular all formulations', 'geography': 'United States', 'unit': 'dollars_per_gallon', 'description': 'U.S. Regular All Formulations Gasoline Price (Weekly)'}),
 ForecastCase(series_id='GASREGW', horizon=1, cutoff_timestamp='2021-03-01 00:00:00', context={'day_of_week': None, 'month': None}, features={'lag_1': 2.633, 'lag_2': 2.501, 'lag_4': 2.409, 'lag_52': 2.423}, actual=2.711, naive_forecast=2.633, seasonal_naive_forecast=None, model_forecast=None, metadata={'protocol': 'one_step_ahead_chronological_no_shuffle', 'metr

In [6]:
case_preview = pd.DataFrame(
    [
        {
            'series_id': c.series_id,
            'cutoff_timestamp': c.cutoff_timestamp,
            'horizon': c.horizon,
            'actual': c.actual,
            'naive_forecast': c.naive_forecast,
            'has_lag_52': 'lag_52' in c.features,
            'source': c.metadata.get('source'),
            'fuel_type': c.metadata.get('fuel_type'),
            'geography': c.metadata.get('geography'),
        }
        for c in cases
    ]
)
case_preview


,series_id,cutoff_timestamp,horizon,actual,naive_forecast,has_lag_52,source,fuel_type,geography
0,GASREGW,2021-02-22 00:00:00,1,2.633,2.501,True,FRED/EIA,regular all formulations,United States
1,GASREGW,2021-03-01 00:00:00,1,2.711,2.633,True,FRED/EIA,regular all formulations,United States
2,GASREGW,2021-03-08 00:00:00,1,2.771,2.711,True,FRED/EIA,regular all formulations,United States
3,GASREGW,2021-03-15 00:00:00,1,2.853,2.771,True,FRED/EIA,regular all formulations,United States
4,GASREGW,2021-03-22 00:00:00,1,2.865,2.853,True,FRED/EIA,regular all formulations,United States


In [7]:
results = [
    result_from_forecast(
        c,
        forecast=float(c.naive_forecast) if c.naive_forecast is not None else 0.0,
        model_name='naive_lag_1',
        notes='Day 1 placeholder result for Week 8 setup',
    )
    for c in cases
]

result_preview = pd.DataFrame(
    [
        {
            'model_name': r.model_name,
            'cutoff_timestamp': r.case.cutoff_timestamp,
            'forecast': r.forecast,
            'actual': r.case.actual,
            'mae_vs_actual': r.mae_vs_actual,
            'smape_vs_actual': r.smape_vs_actual,
        }
        for r in results
    ]
)
result_preview


,model_name,cutoff_timestamp,forecast,actual,mae_vs_actual,smape_vs_actual
0,naive_lag_1,2021-02-22 00:00:00,2.501,2.633,0.132,5.142189
1,naive_lag_1,2021-03-01 00:00:00,2.633,2.711,0.078,2.919162
2,naive_lag_1,2021-03-08 00:00:00,2.711,2.771,0.060,2.188982
3,naive_lag_1,2021-03-15 00:00:00,2.771,2.853,0.082,2.916074
4,naive_lag_1,2021-03-22 00:00:00,2.853,2.865,0.012,0.419727
